# Nailing LLMGPR's Table 1 — the social-edge rule

**Where we are.** `raw_POIs.txt` restricted to New York + Chicago + Los Angeles yields
**exactly 436 categories** — LLMGPR's number — out of a 519-category global vocabulary.
Section 3's entire global dump holds 429, so it was ruled out. Their POI metadata is
Section 5's *raw* dump.

The raw check-ins get close on volume and badly wrong on users:

| | users | POIs (>=1 ck) | check-ins | ck/user |
|---|---|---|---|---|
| raw, unfiltered | 152,480 | 237,728 | 2,227,315 | 14.6 |
| raw, >=10 core | 30,901 | 37,103 | 1,379,959 | 44.7 |
| **LLMGPR** | **7,507** | **80,962** | **1,214,631** | **161.8** |

Same check-in volume as the >=10 core (within 14%), concentrated in 4.1x fewer users. Their
users are ~24% of our core holding ~88% of its check-ins — a heavy, distinctive subpopulation.

**The hypothesis.** LLMGPR builds groups from *social relations*, so it can only use users
who have them. Only 114,324 of the raw dump's 2,733,324 users carry a friendship edge, and
those are exactly the heavy, socially-connected users the WWW2019 subset was selected for.
Restricting to them before the >=10 core should collapse 30,901 users toward 7,507 while
keeping most check-ins — and it also explains `#POIs`: 80,962 is 34% of the 237,728 available
venues, i.e. the venues those retained users actually touched, counted at >=1 check-in.

**This notebook doesn't assume the rule — it sweeps six of them** and prints each as a Table 1
row with ratios, so the numbers pick the winner. It also settles whether the raw and filtered
check-in files share a user-id space, which decides whether friendships attach directly.

Disk peaks ~10.3 GB of 20 GB. Runtime roughly 40-55 min.

## 0. Setup

In [1]:
import os, sys, re, zipfile, subprocess, collections, random, gc
import pandas as pd, numpy as np

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-4000:])
    if p.stderr: print(p.stderr[-4000:], file=sys.stderr)
    if check and p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p

def du(): sh("df -h /kaggle/working | tail -1", check=False)

def find(pat, root=WORK):
    hits=[]
    for dp,_,fns in os.walk(root):
        if "__MACOSX" in dp: continue
        for fn in fns:
            if re.search(pat, fn, re.I) and not fn.startswith("._"): hits.append(os.path.join(dp,fn))
    return sorted(hits)

def save_table(df, stem):
    try: df.to_parquet(f"{stem}.parquet", index=False); out=f"{stem}.parquet"
    except ImportError: df.to_csv(f"{stem}.csv", index=False); out=f"{stem}.csv"
    print("wrote", out); return out

def fetch_drive(file_id, dest, expected_bytes=None, resourcekey=None):
    # gdown's fuzzy parser drops the resourcekey legacy ids need; drive.usercontent honours it
    if os.path.exists(dest) and (expected_bytes is None or os.path.getsize(dest)==expected_bytes):
        print(f"already have {dest} ({os.path.getsize(dest)/1024**3:.2f} GB)")
    else:
        url=(f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
             + (f"&resourcekey={resourcekey}" if resourcekey else ""))
        print("$ curl", url, flush=True)
        rc=subprocess.run(f'curl -L --fail --retry 3 --retry-delay 5 -o "{dest}" "{url}"',shell=True).returncode
        if rc: raise RuntimeError(f"curl failed (exit {rc})")
    with open(dest,"rb") as f: magic=f.read(4)
    assert magic==b"PK\x03\x04", f"{dest} is not a zip (starts {magic!r})"
    print(f"OK {dest}: {os.path.getsize(dest)/1024**3:.2f} GB"); return dest

TARGET_3CITY = dict(users=7_507, pois=80_962, cats=436, ck=1_214_631)
TARGET_NYC   = dict(users=6_078, pois=63_445, cats=436, ck=923_856)
CHUNK = 2_000_000
CITY_BBOX = {
    "New York":    dict(lon_min=-74.3,  lon_max=-73.6,  lat_min=40.4, lat_max=41.0),
    "Chicago":     dict(lon_min=-88.0,  lon_max=-87.5,  lat_min=41.6, lat_max=42.1),
    "Los Angeles": dict(lon_min=-118.7, lon_max=-117.6, lat_min=33.6, lat_max=34.4),
}
CK_COLS  = ["user_id","venue_id","utc_time","tz_offset"]
POI_COLS = ["venue_id","lat","lon","category","country"]

def read_ck(path, **kw):
    return pd.read_csv(path, sep="\t", header=None, names=CK_COLS,
                       dtype={"user_id":str,"venue_id":str,"utc_time":str},
                       usecols=[0,1,2,3], on_bad_lines="skip", **kw)
print("setup ok")

setup ok


## 1. Fetch and extract — raw POIs, raw check-ins, filtered check-ins, friendships

In [2]:
WWW_ZIP=f"{WORK}/dataset_WWW2019.zip"
NEED = dict(raw_pois=r"raw_POIs\.txt$", raw_ck=r"raw_Checkins.*\.txt$",
            filt_ck=r"WWW_Checkins.*\.txt$", f_old=r"friendship_old.*\.txt$",
            f_new=r"friendship_new.*\.txt$")
paths={k:find(v) for k,v in NEED.items()}

if not all(paths.values()):
    fetch_drive("1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8", WWW_ZIP, expected_bytes=2_684_000_558)
    with zipfile.ZipFile(WWW_ZIP) as z:
        wanted=[n for n in z.namelist() if "__MACOSX" not in n and not n.endswith("/")]
        print("extracting all", len(wanted), "members (~7.8 GB, several minutes)")
        for n in wanted: z.extract(n, WORK); print("  done", n, flush=True)
    os.remove(WWW_ZIP)
    paths={k:find(v) for k,v in NEED.items()}

P={k:v[0] for k,v in paths.items()}
for k,v in P.items(): print(f"{k:<9}: {v}  ({os.path.getsize(v)/1024**3:.2f} GB)")
du()

$ curl https://drive.usercontent.google.com/download?id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2559M  100 2559M    0     0   150M      0  0:00:17  0:00:17 --:--:--  163M


OK /kaggle/working/dataset_WWW2019.zip: 2.50 GB
extracting all 6 members (~7.8 GB, several minutes)
  done dataset_WWW2019/raw_Checkins_anonymized.txt
  done dataset_WWW2019/raw_POIs.txt
  done dataset_WWW2019/dataset_WWW_friendship_old.txt
  done dataset_WWW2019/dataset_WWW_readme.txt
  done dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt
  done dataset_WWW2019/dataset_WWW_friendship_new.txt
raw_pois : /kaggle/working/dataset_WWW2019/raw_POIs.txt  (0.66 GB)
raw_ck   : /kaggle/working/dataset_WWW2019/raw_Checkins_anonymized.txt  (5.69 GB)
filt_ck  : /kaggle/working/dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt  (1.44 GB)
f_old    : /kaggle/working/dataset_WWW2019/dataset_WWW_friendship_old.txt  (0.00 GB)
f_new    : /kaggle/working/dataset_WWW2019/dataset_WWW_friendship_new.txt  (0.01 GB)
$ df -h /kaggle/working | tail -1
/dev/loop1       20G  7.8G   12G  40% /kaggle/working



## 2. City venue sets and the category vocabulary (re-derived, cheap)

In [3]:
city_venues={c:set() for c in CITY_BBOX}; city_cats={c:set() for c in CITY_BBOX}
VENUE2CAT={}; seen=0
for ch in pd.read_csv(P["raw_pois"], sep="\t", header=None, names=POI_COLS,
                      dtype={"venue_id":str,"category":str}, on_bad_lines="skip", chunksize=CHUNK):
    seen+=len(ch)
    ch["lat"]=pd.to_numeric(ch["lat"],errors="coerce"); ch["lon"]=pd.to_numeric(ch["lon"],errors="coerce")
    ch=ch.dropna(subset=["lat","lon"])
    for city,b in CITY_BBOX.items():
        sub=ch[ch["lon"].between(b["lon_min"],b["lon_max"]) & ch["lat"].between(b["lat_min"],b["lat_max"])]
        if len(sub):
            city_venues[city]|=set(sub["venue_id"]); city_cats[city]|=set(sub["category"].dropna().unique())
            VENUE2CAT.update(zip(sub["venue_id"],sub["category"]))
    print(f"\rscanned {seen:,} POIs", end="", flush=True)
CATS_3CITY=set().union(*city_cats.values())
KEEP=pd.Index(sorted(set().union(*city_venues.values())))
print(f"\n3-city venues {len(KEEP):,} | categories {len(CATS_3CITY):,} (LLMGPR: {TARGET_3CITY['cats']})")
VENUE2CITY={v:c for c,vs in city_venues.items() for v in vs}

scanned 11,180,160 POIs
3-city venues 237,728 | categories 436 (LLMGPR: 436)


## 3. Both check-in files, restricted to the three cities

In [4]:
def scan(path, label):
    parts,seen=[],0
    for ch in read_ck(path, chunksize=CHUNK):
        seen+=len(ch); parts.append(ch[ch["venue_id"].isin(KEEP)])
        print(f"\r{label}: scanned {seen:,}", end="", flush=True)
    d=pd.concat(parts,ignore_index=True); del parts; gc.collect()
    print(f"\n{label}: {len(d):,} in-scope check-ins, {d['user_id'].nunique():,} users")
    return d

raw  = scan(P["raw_ck"],  "raw     ")
filt = scan(P["filt_ck"], "filtered")

raw     : scanned 90,048,627
raw     : 2,227,756 in-scope check-ins, 152,480 users
filtered: scanned 22,809,624
filtered: 592,341 in-scope check-ins, 14,401 users


## 4. Do the raw and filtered files share a user-id space?

This decides whether the 114,324 friendship users can be named in raw check-ins at all.
Both 3-city subsets are already in memory, so the fingerprint match is cheap — no rescan.

In [5]:
direct = len(set(filt["user_id"].unique()) & set(raw["user_id"].unique()))
print(f"raw 3-city users      : {raw['user_id'].nunique():,}")
print(f"filtered 3-city users : {filt['user_id'].nunique():,}")
print(f"literal id intersection: {direct:,}\n")

fp = (filt[["user_id","venue_id","utc_time"]].rename(columns={"user_id":"f_user"})
        .drop_duplicates(subset=["venue_id","utc_time"]))
m = raw.merge(fp, on=["venue_id","utc_time"], how="inner")
print(f"fingerprint hits: {len(m):,}")

votes=collections.defaultdict(collections.Counter)
for f,r in zip(m["f_user"], m["user_id"]): votes[f][r]+=1
matched=[u for u in votes if votes[u]]
identical=sum(1 for u in matched if votes[u].most_common(1)[0][0]==u)
purity=sorted(votes[u].most_common(1)[0][1]/sum(votes[u].values()) for u in matched)
med=purity[len(purity)//2] if purity else 0.0
print(f"filtered users located in raw: {len(matched):,} / {filt['user_id'].nunique():,}")
print(f"  ...with the SAME id        : {identical:,}")
print(f"  median vote purity         : {med:.2f}")

if matched and identical >= 0.9*len(matched):
    FILT2RAW = {u:u for u in matched}
    print("\n=> SHARED ID SPACE. friendship ids name raw users directly.")
else:
    FILT2RAW = {u: votes[u].most_common(1)[0][0] for u in matched
                if votes[u].most_common(1)[0][1] >= 0.9*sum(votes[u].values())}
    print(f"\n=> DISTINCT ids; recovered a clean 1:1 map for {len(FILT2RAW):,} users.")

raw 3-city users      : 152,480
filtered 3-city users : 14,401
literal id intersection: 14,401

fingerprint hits: 592,385
filtered users located in raw: 14,400 / 14,401
  ...with the SAME id        : 14,400
  median vote purity         : 1.00

=> SHARED ID SPACE. friendship ids name raw users directly.


## 5. The friendship user set, in raw id space

In [6]:
def edges(path):
    e=pd.read_csv(path, sep="\t", header=None, names=["u","v"], dtype=str)
    return set(e["u"])|set(e["v"])

F_OLD = edges(P["f_old"]); F_ALL = F_OLD | edges(P["f_new"])
print(f"users with a friendship_old edge : {len(F_OLD):,}")
print(f"users with any friendship edge   : {len(F_ALL):,}")

FRIENDS_OLD = {FILT2RAW[u] for u in F_OLD if u in FILT2RAW}
FRIENDS_ALL = {FILT2RAW[u] for u in F_ALL if u in FILT2RAW}
raw_users = set(raw["user_id"].unique())
print(f"\nfriendship_old users present in raw 3-city: {len(FRIENDS_OLD & raw_users):,}")
print(f"any-friendship users present in raw 3-city: {len(FRIENDS_ALL & raw_users):,}")
print(f"  (of {len(raw_users):,} raw 3-city users)")

users with a friendship_old edge : 114,324
users with any friendship edge   : 114,324

friendship_old users present in raw 3-city: 14,400
any-friendship users present in raw 3-city: 14,400
  (of 152,480 raw 3-city users)


## 6. Rule sweep

Six candidate readings of *"users and POIs with less than 10 interactions are removed"*,
each reported as a Table 1 row. `#POIs` counts venues with >=1 check-in among retained users
— their convention, since their column is provably not a post-filter count.

In [7]:
def k_core(df, k=10, user_only=False):
    while True:
        n=len(df)
        vc=df["user_id"].value_counts();  df=df[df["user_id"].isin(vc[vc>=k].index)]
        if not user_only:
            vc=df["venue_id"].value_counts(); df=df[df["venue_id"].isin(vc[vc>=k].index)]
        if len(df)==n or df.empty: return df

def row(name, d, target=TARGET_3CITY):
    if d.empty: print(f"{name:<34}{'(empty)':>10}"); return
    u=d["user_id"].nunique(); p=d["venue_id"].nunique(); n=len(d)
    c=pd.Series([VENUE2CAT.get(v) for v in d["venue_id"].unique()]).nunique()
    score=np.mean([min(a,b)/max(a,b) for a,b in
                   ((u,target["users"]),(p,target["pois"]),(c,target["cats"]),(n,target["ck"]))])
    print(f"{name:<34}{u:>9,}{p:>10,}{c:>6}{n:>12,}{n/u:>9.1f}{score:>8.2f}")

hdr=f"{'rule':<34}{'users':>9}{'POIs>=1':>10}{'cats':>6}{'check-ins':>12}{'ck/user':>9}{'match':>8}"
print(hdr); print("-"*len(hdr))
row("A  no filter", raw)
row("B  >=10 core", k_core(raw.copy()))
row("C  >=10 users only", k_core(raw.copy(), user_only=True))
row("D  friends(old), no filter", raw[raw["user_id"].isin(FRIENDS_OLD)])
row("E  friends(old) + >=10 core", k_core(raw[raw["user_id"].isin(FRIENDS_OLD)].copy()))
row("F  friends(any) + >=10 core", k_core(raw[raw["user_id"].isin(FRIENDS_ALL)].copy()))
print("-"*len(hdr))
t=TARGET_3CITY
print(f"{'TARGET (CIKM 3 cities)':<34}{t['users']:>9,}{t['pois']:>10,}{t['cats']:>6}{t['ck']:>12,}"
      f"{t['ck']/t['users']:>9.1f}{1.00:>8.2f}")
print("\n'match' = mean of min/max ratio across all four columns; 1.00 is exact.")

rule                                  users   POIs>=1  cats   check-ins  ck/user   match
----------------------------------------------------------------------------------------
A  no filter                        152,480   237,728   436   2,227,756     14.6    0.48
B  >=10 core                         30,907    37,113   400   1,380,405     44.7    0.62
C  >=10 users only                   38,500   218,103   432   1,901,636     49.4    0.55
D  friends(old), no filter           14,400   102,541   427     592,340     41.1    0.69
E  friends(old) + >=10 core           5,116    11,800   355     349,519     68.3    0.48
F  friends(any) + >=10 core           5,116    11,800   355     349,519     68.3    0.48
----------------------------------------------------------------------------------------
TARGET (CIKM 3 cities)                7,507    80,962   436   1,214,631    161.8    1.00

'match' = mean of min/max ratio across all four columns; 1.00 is exact.


## 7. Same sweep, New York only (against the arXiv v1 table)

In [8]:
ny = raw[raw["venue_id"].map(VENUE2CITY).eq("New York")]
print(hdr); print("-"*len(hdr))
row("A  no filter",                ny, TARGET_NYC)
row("B  >=10 core",                k_core(ny.copy()), TARGET_NYC)
row("E  friends(old) + >=10 core", k_core(ny[ny["user_id"].isin(FRIENDS_OLD)].copy()), TARGET_NYC)
row("F  friends(any) + >=10 core", k_core(ny[ny["user_id"].isin(FRIENDS_ALL)].copy()), TARGET_NYC)
print("-"*len(hdr))
t=TARGET_NYC
print(f"{'TARGET (arXiv v1, NYC)':<34}{t['users']:>9,}{t['pois']:>10,}{t['cats']:>6}{t['ck']:>12,}"
      f"{t['ck']/t['users']:>9.1f}{1.00:>8.2f}")

rule                                  users   POIs>=1  cats   check-ins  ck/user   match
----------------------------------------------------------------------------------------
A  no filter                         88,853   113,326   433   1,164,743     13.1    0.60
B  >=10 core                         16,783    18,453   379     728,984     43.4    0.58
E  friends(old) + >=10 core           2,843     5,955   317     181,945     64.0    0.37
F  friends(any) + >=10 core           2,843     5,955   317     181,945     64.0    0.37
----------------------------------------------------------------------------------------
TARGET (arXiv v1, NYC)                6,078    63,445   436     923,856    152.0    1.00


## 8. Emit the winning dataset

In [10]:
# Change WINNER to whichever rule scored highest above, then re-run this cell.
WINNER = "A"
sel = {"A": raw,
       "B": k_core(raw.copy()),
       "C": k_core(raw.copy(), user_only=True),
       "D": raw[raw["user_id"].isin(FRIENDS_OLD)],
       "E": k_core(raw[raw["user_id"].isin(FRIENDS_OLD)].copy()),
       "F": k_core(raw[raw["user_id"].isin(FRIENDS_ALL)].copy())}[WINNER]

sel = sel.assign(city=sel["venue_id"].map(VENUE2CITY),
                 category=sel["venue_id"].map(VENUE2CAT))
save_table(sel, f"{WORK}/llmgpr_checkins_{WINNER}")

users = set(sel["user_id"].unique())
eo = pd.read_csv(P["f_old"], sep="\t", header=None, names=["u","v"], dtype=str)
eo = eo.assign(u=eo["u"].map(FILT2RAW), v=eo["v"].map(FILT2RAW)).dropna()
eo = eo[eo["u"].isin(users) & eo["v"].isin(users)]
save_table(eo, f"{WORK}/llmgpr_friendship_old")
print(f"\nfriendship_old edges among retained users: {len(eo):,}")
print("friendship_NEW is deliberately excluded - it postdates the check-in window and leaks.")

pois = (pd.DataFrame({"venue_id": sorted(set(sel['venue_id']))})
          .assign(city=lambda d: d["venue_id"].map(VENUE2CITY),
                  category=lambda d: d["venue_id"].map(VENUE2CAT)))
save_table(pois, f"{WORK}/llmgpr_pois")
print(f"\nfinal: {len(users):,} users | {len(pois):,} POIs | {len(sel):,} check-ins | {len(eo):,} edges")

wrote /kaggle/working/llmgpr_checkins_A.parquet
wrote /kaggle/working/llmgpr_friendship_old.parquet

friendship_old edges among retained users: 22,471
friendship_NEW is deliberately excluded - it postdates the check-in window and leaks.
wrote /kaggle/working/llmgpr_pois.parquet

final: 152,480 users | 237,728 POIs | 2,227,756 check-ins | 22,471 edges


In [11]:
for p in (P["raw_ck"], P["raw_pois"], P["filt_ck"]):
    if os.path.exists(p): os.remove(p); print("removed", p)
du()

removed /kaggle/working/dataset_WWW2019/raw_Checkins_anonymized.txt
removed /kaggle/working/dataset_WWW2019/raw_POIs.txt
removed /kaggle/working/dataset_WWW2019/dataset_WWW_Checkins_anonymized.txt
$ df -h /kaggle/working | tail -1
/dev/loop1       20G   76M   20G   1% /kaggle/working



## What to send back

The **rule sweep** table from section 6 (and section 7), plus the id-space verdict from
section 4. The `match` column ranks the rules; whichever tops it is LLMGPR's preprocessing,
and section 8 emits that dataset — check-ins, POIs with city and category, and
`friendship_old` edges restricted to the retained users.

If no rule clears ~0.85, their table isn't reproducible even from the right dump, and we
adopt the closest rule, report our own numbers, and note the discrepancy. Either way the
next step is group construction: persistent member-sets with pooled sequences, a rolling
time window, cliques rather than connected components, and leave-one-out — then GBSR.